# router

> Tier 2 convenience router with auto-wired reorder, remove, and clear endpoints.

In [ ]:
#| default_exp router

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Any, Callable, List, Optional, Tuple

from fasthtml.common import APIRouter

from cjm_fasthtml_sortable_queue.config import SortableQueueConfig
from cjm_fasthtml_sortable_queue.html_ids import SortableQueueHtmlIds
from cjm_fasthtml_sortable_queue.models import SortableQueueUrls
from cjm_fasthtml_sortable_queue.handlers import handle_reorder, handle_remove, handle_clear

In [ ]:
#| export
def init_sortable_queue_router(
    config: SortableQueueConfig,  # Queue configuration
    get_items: Callable[[str], List[dict]],  # (session_id) -> current items
    set_items: Callable[[str, List[dict]], None],  # (session_id, items) -> persist
    render_content: Callable[[dict, int], Any],  # Custom content callback
    on_mutate: Optional[Callable[[str, List[dict], str], tuple]] = None,  # (mutation_type, items, sess) -> OOB elements
    prefix: str = "/queue",  # Route prefix
    **render_kwargs,  # Additional kwargs passed to render_sortable_queue
) -> Tuple[APIRouter, SortableQueueUrls]:  # (router, urls) tuple
    """Initialize a router with reorder, remove, and clear endpoints.

    The `on_mutate` callback receives the mutation type ("reorder", "remove",
    "clear"), the updated items list, and the session ID. It should return a
    tuple of OOB elements to append to the response, or an empty tuple.
    """
    router = APIRouter(prefix=prefix)
    ids = SortableQueueHtmlIds(prefix=config.prefix)

    # Build URLs after router creation (prefix is applied)
    urls = SortableQueueUrls(
        reorder=f"{prefix}/reorder",
        remove=f"{prefix}/remove",
        clear=f"{prefix}/clear",
    )

    @router
    async def reorder(request, sess):
        """Handle Sortable.js drag-end reorder."""
        form_data = await request.form()
        new_key_order = form_data.getlist("item")
        items = get_items(sess)
        updated, panel = handle_reorder(config, ids, urls, items, new_key_order, render_content, **render_kwargs)
        set_items(sess, updated)
        if on_mutate:
            oob = on_mutate("reorder", updated, sess)
            if oob:
                return (panel, *oob)
        return panel

    @router
    async def remove(request, sess, key: str = ""):
        """Handle item removal."""
        items = get_items(sess)
        updated, panel = handle_remove(config, ids, urls, items, key, render_content, **render_kwargs)
        set_items(sess, updated)
        if on_mutate:
            oob = on_mutate("remove", updated, sess)
            if oob:
                return (panel, *oob)
        return panel

    @router
    async def clear(sess):
        """Handle clear all."""
        updated, panel = handle_clear(config, ids, urls, render_content, **render_kwargs)
        set_items(sess, updated)
        if on_mutate:
            oob = on_mutate("clear", updated, sess)
            if oob:
                return (panel, *oob)
        return panel

    return router, urls

## Tests

In [ ]:
from fasthtml.common import Span

# Test setup
config = SortableQueueConfig(prefix="rt")
state = {"items": [{"id": "a"}, {"id": "b"}]}

def get_items(sess):
    return list(state["items"])

def set_items(sess, items):
    state["items"] = items

def content_fn(item, index):
    return Span(item["id"])

# --- Router creation ---
router, urls = init_sortable_queue_router(
    config=config,
    get_items=get_items,
    set_items=set_items,
    render_content=content_fn,
    prefix="/test-queue",
)

# Verify return types
assert isinstance(router, APIRouter)
assert isinstance(urls, SortableQueueUrls)

# Verify URL paths
assert urls.reorder == "/test-queue/reorder"
assert urls.remove == "/test-queue/remove"
assert urls.clear == "/test-queue/clear"

# --- With on_mutate callback ---
mutate_log = []

def on_mutate(mutation_type, items, sess):
    mutate_log.append((mutation_type, len(items)))
    return ()

router2, urls2 = init_sortable_queue_router(
    config=config,
    get_items=get_items,
    set_items=set_items,
    render_content=content_fn,
    on_mutate=on_mutate,
    prefix="/q2",
)
assert urls2.reorder == "/q2/reorder"

print("All router tests passed")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()